# EXP008_READINESS_V1 — frozen-weight replay and nominal-calibration feasibility

**Enable A100 GPU → Run all → provide the input bundle once.** No training, final TRAIN calibration, NASA TEST or anomaly labels. All results are descriptive. This workflow deliberately stops without selecting a detector.

Read START_HERE.md and PROTOCOL.md first. Exact parent CUDA environment and numerical replay are mandatory gates. Do not edit cells/settings to bypass failures. The executed CUDA workflow has not been run during preparation; local engineering tests are included in the bundle.


In [ ]:
import os, sys, json, hashlib, subprocess, shutil, time, zipfile
from pathlib import Path, PurePosixPath
from google.colab import drive, files
EXPECTED_BUNDLE_SHA256 = "60304e17d166dd9dc2d2ba65c961ffd5448637a52cc4e185caa95089ddd0397b"
BUNDLE_NAME = "EXP008_READINESS_V1_input_bundle.zip"
subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version", "--format=csv"], check=True)
drive.mount("/content/drive")
BASE=Path("/content/drive/MyDrive/NASA_GCN_EXP008_READINESS_V1")
OUTPUT=BASE/"results"; BASE.mkdir(parents=True,exist_ok=True); OUTPUT.mkdir(exist_ok=True)
lease=OUTPUT/"ACTIVE_RUN.json"
if lease.exists() and time.time()-json.loads(lease.read_text())["heartbeat"]<600:
    raise RuntimeError("A run may still be active. Stop the old runtime and wait ten minutes after its heartbeat. Do not delete the lease.")
def file_sha(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):h.update(b)
    return h.hexdigest()
DURABLE=BASE/BUNDLE_NAME
if not DURABLE.exists():
    print("Upload the supplied "+BUNDLE_NAME)
    uploaded=files.upload()
    if len(uploaded)!=1:raise RuntimeError("Exactly one input bundle required")
    name,data=next(iter(uploaded.items()))
    if hashlib.sha256(data).hexdigest()!=EXPECTED_BUNDLE_SHA256:raise RuntimeError("Wrong bundle")
    tmp=DURABLE.with_suffix(".uploading")
    with open(tmp,"wb") as f:f.write(data);f.flush();os.fsync(f.fileno())
    if file_sha(tmp)!=EXPECTED_BUNDLE_SHA256:raise RuntimeError("Upload checksum failed")
    os.replace(tmp,DURABLE)
if file_sha(DURABLE)!=EXPECTED_BUNDLE_SHA256:raise RuntimeError("Durable bundle changed")
WORK=Path("/content/exp008_readiness_v1_work");ROOT=WORK/"package";ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(DURABLE) as z:
    if len(z.namelist())!=len(set(z.namelist())):raise RuntimeError("Duplicate members")
    for info in z.infolist():
        p=PurePosixPath(info.filename)
        if p.is_absolute() or ".." in p.parts or (info.external_attr>>16)&0o170000==0o120000:raise RuntimeError("Unsafe archive member")
    z.extractall(ROOT)
manifest=json.loads((ROOT/"PACKAGE_MANIFEST.json").read_text())
actual={str(p.relative_to(ROOT)) for p in ROOT.rglob("*") if p.is_file() and p.name!="PACKAGE_MANIFEST.json" and "__pycache__" not in p.parts}
if actual!=set(manifest["files"]):raise RuntimeError("Input file roster mismatch")
for rel,h in manifest["files"].items():
    if file_sha(ROOT/rel)!=h:raise RuntimeError("Changed input: "+rel)
parent=json.loads((ROOT/"provenance/RUN_FREEZE.json").read_text())
import platform
if platform.python_version()!=parent["environment"]["python"]:
    raise RuntimeError("Python differs from parent. Preserve the mismatch and request a versioned portability review, not a silent change.")
print("Verified immutable bundle. Final20% TRAIN and NASA TEST/labels are absent.")


In [ ]:
# Use virtualenv directly: avoid the previous Colab stdlib ensurepip failure.
LOGS=OUTPUT/"logs";LOGS.mkdir(exist_ok=True)
setup_log=LOGS/(time.strftime("setup_%Y%m%dT%H%M%SZ",time.gmtime())+".log")
def logged(command,env=None):
    with open(setup_log,"a",buffering=1) as log:
        log.write("COMMAND "+repr(command)+"\n")
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        try:
            for line in process.stdout:print(line,end="");log.write(line)
            code=process.wait()
        except BaseException:
            process.terminate()
            try:process.wait(timeout=30)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        if code:raise RuntimeError("Provisioning failed. See "+str(setup_log))
ENV=WORK/"venv";PYTHON=ENV/"bin/python";READY=ENV/"EXP008_READY.json"
if not PYTHON.exists():
    logged([sys.executable,"-m","pip","install","--disable-pip-version-check","virtualenv"])
    logged([sys.executable,"-m","virtualenv",str(ENV)])
LOCK=ROOT/"provenance/requirements-resolved.txt"
if not READY.exists():
    logged([str(PYTHON),"-m","pip","install","--disable-pip-version-check","-r",str(LOCK),"--extra-index-url","https://download.pytorch.org/whl/cu130"])
logged([str(PYTHON),"-m","pip","check"])
resolved=subprocess.check_output([str(PYTHON),"-m","pip","freeze","--all"],text=True)
if sorted(resolved.splitlines())!=sorted(LOCK.read_text().splitlines()):
    (OUTPUT/"ACTUAL_REQUIREMENTS_MISMATCH.txt").write_text(resolved)
    raise RuntimeError("Pinned environment mismatch. Do not proceed.")
READY.write_text(json.dumps({"requirements_sha256":file_sha(LOCK)}))
(OUTPUT/"requirements-resolved.txt").write_text(resolved)
RUN_ENV=os.environ.copy();RUN_ENV.update(PYTHONHASHSEED="17",CUBLAS_WORKSPACE_CONFIG=":4096:8",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",OPENBLAS_NUM_THREADS="1",PYTHONUNBUFFERED="1",PYTHONDONTWRITEBYTECODE="1")
print("Isolated dependencies verified. The runner also checks the complete parent GPU/environment fingerprint.")


In [ ]:
log_path=LOGS/(time.strftime("run_%Y%m%dT%H%M%SZ",time.gmtime())+".log")
command=[str(PYTHON),"-u",str(ROOT/"src/run_exp008.py"),"--output",str(OUTPUT)]
with open(log_path,"a",buffering=1) as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=RUN_ENV)
    try:
        for line in process.stdout:print(line,end="");log.write(line);log.flush()
        returncode=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=30)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
if returncode:raise RuntimeError("Stopped safely; committed replays remain on Drive. See "+str(log_path))
status=json.loads((OUTPUT/"STATUS.json").read_text())
if status.get("status")!="COMPLETE_NO_DETECTOR_SELECTED" or status.get("committed")!=534:raise RuntimeError("Completeness gate failed")
print((OUTPUT/"SUMMARY.md").read_text())


In [ ]:
from IPython.display import display,Markdown
print("Completed. No detector has been selected and no final evaluation is authorised.")
display(Markdown((OUTPUT/"SUMMARY.md").read_text()))
for p in sorted((OUTPUT/"exports").glob("*.zip")):print(p.name,round(p.stat().st_size/1e6,1),"MB",str(p))
print("Save a copy of this executed notebook for provenance. Send both result exports for audit.")
try:files.download(str(OUTPUT/"exports/EXP008_summary.zip"))
except Exception as exc:print("Download unavailable; both exports remain on Drive:",exc)
